## Model Creation and Evaluation

In [13]:
# import packages 
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [3]:
df = pd.read_csv('../data/cleaned_data.csv') 

# convert the column to datetime objects
df['start_time'] = pd.to_datetime(df['start_time'])

# put hour into its own column
df['hour'] = df['start_time'].dt.hour

In [6]:
X = pd.get_dummies(df[['state', 'Event Type', 'mean_customers', 'hour']], drop_first=True)
y = df['duration']

# Split before scaling to avoid data leakage
X_train_unscaled, X_test_unscaled, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create scaled copies (for models sensitive to feature magnitude)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_unscaled)
X_test_scaled = scaler.transform(X_test_unscaled)

# Convert back to DataFrame for convenience (especially with SHAP or plotting)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train_unscaled.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test_unscaled.index)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [35]:
model = LinearRegression()

# Cross validation
cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2').mean()

# Train
model.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = model.predict(X_test_scaled)

# Metrics
r2 = r2_score(y_test, y_pred_lr)
mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("Linear Regression Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Linear Regression Results
CV R2: 0.2778374524877302
Test R2: 0.11765701329735856
MAE (Hours): 6.482758125899894
RMSE (Hours): 14.726673780344296


In [36]:
model = RandomForestRegressor(n_estimators=100, random_state=42)

cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2').mean()

model.fit(X_train_scaled, y_train)

y_pred_rf = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred_rf)
mae = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("Random Forest Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Random Forest Results
CV R2: 0.9432175021150186
Test R2: 0.9492902443953755
MAE (Hours): 0.9603875359436345
RMSE (Hours): 3.5304633410503015


In [37]:
model = GradientBoostingRegressor(n_estimators=100, random_state=42)

cv_r2 = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='r2').mean()

model.fit(X_train_scaled, y_train)

y_pred_gb = model.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred_gb)
mae = mean_absolute_error(y_test, y_pred_gb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))

print("Gradient Boosting Results")
print("CV R2:", cv_r2)
print("Test R2:", r2)
print("MAE (Hours):", mae)
print("RMSE (Hours):", rmse)

Gradient Boosting Results
CV R2: 0.6377985135993207
Test R2: 0.6403464404452137
MAE (Hours): 4.74181544683686
RMSE (Hours): 9.4021706424414


## Hyperparameter Tuning

In [38]:
# Parameter grid
lr_param_grid = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

# Initialize model
lr = LinearRegression()

# Grid Search
lr_grid = GridSearchCV(
    lr,
    lr_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# Fit
lr_grid.fit(X_train_scaled, y_train)

# Best model
best_lr = lr_grid.best_estimator_

# Predictions
y_pred_lr_tuned = best_lr.predict(X_test_scaled)

# Metrics
print("Best Parameters:", lr_grid.best_params_)
print("Test R2:", r2_score(y_test, y_pred_lr_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_lr_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr_tuned)))

Best Parameters: {'fit_intercept': True, 'positive': False}
Test R2: 0.11765701329735856
MAE: 6.482758125899894
RMSE: 14.726673780344296


In [34]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)

rf_random = RandomizedSearchCV(
    rf,
    rf_param_grid,
    n_iter=10,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

rf_random.fit(X_train_scaled, y_train)

best_rf = rf_random.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test_scaled)

print("Best Parameters:", rf_random.best_params_)
print("Test R2:", r2_score(y_test, y_pred_rf_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_rf_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned)))

Best Parameters: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': None}
Test R2: 0.9505538067310725
MAE: 0.9801034304840045
RMSE: 3.4862006425170993


In [39]:
gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}

gb = GradientBoostingRegressor(random_state=42)

gb_random = RandomizedSearchCV(
    gb,
    gb_param_grid,
    n_iter=12,        
    cv=3,             
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

gb_random.fit(X_train_scaled, y_train)

best_gb = gb_random.best_estimator_
y_pred_gb_tuned = best_gb.predict(X_test_scaled)

print("Best Parameters:", gb_random.best_params_)
print("Test R2:", r2_score(y_test, y_pred_gb_tuned))
print("MAE:", mean_absolute_error(y_test, y_pred_gb_tuned))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gb_tuned)))
gb = GradientBoostingRegressor(random_state=42)

Best Parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Test R2: 0.8730580343840617
MAE: 3.1047705832321064
RMSE: 5.585840839904825


In [40]:
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Assign predictions from your code
y_pred_lr_before = y_pred_lr  # after first Linear Regression run
y_pred_rf_before = y_pred_rf  # after first Random Forest run
y_pred_gb_before = y_pred_gb # after first Gradient Boosting run

y_pred_lr_after = y_pred_lr_tuned
y_pred_rf_after = y_pred_rf_tuned
y_pred_gb_after = y_pred_gb_tuned

models = ["Linear Regression", "Random Forest", "Gradient Boosting"]
rows = []

for model, y_before, y_after in zip(
    models,
    [y_pred_lr_before, y_pred_rf_before, y_pred_gb_before],
    [y_pred_lr_after, y_pred_rf_after, y_pred_gb_after]
):
    rows.append({
        "Model": model,
        "R2 Before": r2_score(y_test, y_before),
        "R2 After": r2_score(y_test, y_after),
        "MAE Before": mean_absolute_error(y_test, y_before),
        "MAE After": mean_absolute_error(y_test, y_after),
        "RMSE Before": np.sqrt(mean_squared_error(y_test, y_before)),
        "RMSE After": np.sqrt(mean_squared_error(y_test, y_after))
    })

performance_table = pd.DataFrame(rows)
print(performance_table)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 2))  
ax.axis('tight')
ax.axis('off')

# Create a table in the plot
table = ax.table(cellText=performance_table.values,
                 colLabels=performance_table.columns,
                 cellLoc='center',
                 loc='center')

plt.savefig("../outputs/performance_table.png", dpi=150)
plt.close()

               Model  R2 Before  R2 After  MAE Before  MAE After  RMSE Before  \
0  Linear Regression   0.117657  0.117657    6.482758   6.482758    14.726674   
1      Random Forest   0.949290  0.950554    0.960388   0.980103     3.530463   
2  Gradient Boosting   0.640346  0.873058    4.741815   3.104771     9.402171   

   RMSE After  
0   14.726674  
1    3.486201  
2    5.585841  
